# Microsoft Fabric Notebook: 04_transform_customers
**Target Lakehouse**: `Silver_Lakehouse`  
**Target Table**: `silver_customers` (Delta Lake)  
**Description**: Cleanses, standardizes types, removes nulls, deduplicates, and writes to Silver Delta table.


In [ ]:
# Fabric Notebook Parameters Cell (Configurable via Fabric Data Factory Pipelines)
pipeline_run_id = "RUN_FABRIC_20260908"
environment = "PROD"
source_system = "FABRIC_INGEST_ENGINE"

In [ ]:
from pyspark.sql.functions import col, trim, lower, to_date, current_timestamp

# Read Raw Customers from Bronze Lakehouse
bronze_customers = spark.read.table("Bronze_Lakehouse.raw_customers")

# Cleanse, Standardize, and Deduplicate
silver_df = bronze_customers.filter(col("customer_id").isNotNull() & (trim(col("customer_id")) != "")) \
    .withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn("first_name", trim(col("first_name"))) \
    .withColumn("last_name", trim(col("last_name"))) \
    .withColumn("full_name", trim(col("first_name")) + " " + trim(col("last_name"))) \
    .withColumn("email", lower(trim(col("email")))) \
    .withColumn("gender", trim(col("gender"))) \
    .withColumn("city", trim(col("city"))) \
    .withColumn("state", trim(col("state"))) \
    .withColumn("country", trim(col("country"))) \
    .withColumn("registration_date", to_date(col("registration_date"), "yyyy-MM-dd")) \
    .withColumn("customer_segment", trim(col("customer_segment"))) \
    .withColumn("updated_timestamp", current_timestamp()) \
    .dropDuplicates(["customer_id"])

# Write to Silver Lakehouse Delta Table
silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_customers")
print(f"[FABRIC SILVER] Transformed {silver_df.count()} records into silver_customers Delta table.")
